# Phase 12 - Eichung des Ablations-Instruments

**Braucht eine A100**, ~20 min.

Der FFN-Lauf v2 endete mit `KEIN-EFFEKT`: 64 ausgewaehlte Einheiten auf null zu setzen
bewegte die Kipprate nicht (70.3% -> 81.2%, p=0.22), gleich viele zufaellige auch nicht
(75.0%, p=0.69). Die Nachpruefungen waren sauber - der Eingriff hat gegriffen. Trotzdem
sagt dieser Befund **nichts ueber das Modell**, solange eine Frage offen ist:

> Kann das Ausschalten einzelner FFN-Einheiten in diesem Modell ueberhaupt ein
> Verhalten bewegen?

Ist die Antwort nein, dann ist jeder Nullbefund - auch einer mit besserer Auswahl - ein
Befund ueber das Werkzeug und nicht ueber die Sache. Diese Zelle verfeinert deshalb
nicht die Auswahl, sie **eicht das Instrument**.

## Zwei Defekte aus v2, die hier behoben sind

**`Trennwerte 31.50 .. 11523438.00`.** In `trenn_statistik` stand
`sd = sqrt(pooled_var) + 1e-8`. Eine Einheit mit Varianz null in *beiden* Haelften -
konstant in den hohen Praefixen, exakt 0 in den niedrigen, weil der Router ihren Experten
dort nicht waehlt - bekam den Nenner `1e-8` und damit einen Trennwert von `1e7`. Gewaehlt
wurden also Einheiten mit *verschwindender Streuung*, nicht mit grosser Trennung. Hier
wird auf **einen globalen, robusten Massstab** normiert (Median der Betraege ueber alle
Einheiten, Nullen ausgenommen). Der kann nicht kollabieren.

**`aktive (Schicht,Experte)-Paare: 1010 | in JEDEM Praefix: 13`.** Von 1010 Paaren waren
13 ueberall aktiv, der Rest der Merkmalsmatrix war **strukturelle Null aus
unterschiedlichem Routing**. Die Auswahl mass "dieser Experte feuert hier und dort nicht"
- bei fuenf verschiedenen Zeichenketten trivial wahr und ohne Bezug zum Verhalten. Hier
werden nur Paare benutzt, die in **allen** verglichenen Zustaenden aktiv sind.

## Der Aufbau

**A - Positivkontrolle.** Zwei Arme desselben Prompts mit einem Verhaltensabstand, der
nicht strittig ist:

| Arm | Phrase | erwartet |
|---|---|---|
| JP | `each service's Japanese name` | Antwort in japanischer Schrift |
| NEU | `each service's name` | Antwort englisch |

Einheiten aus der Zustandsdifferenz waehlen (im JP-Zustand hoeher), auf null setzen,
**Japanisch-Rate** neu messen. Kontrolle: gleich viele zufaellige Einheiten aus derselben
gemeinsamen Menge. Zwei Dosen, 64 und 512. Das ist die Eichmarke - eine Anweisung, der
das Modell zu ~100% folgt.

**B - Dosisleiter.** Am urspruenglichen Prompt 64/256/1024 *zufaellige* aktive Einheiten
ausschalten, neben der Kipprate ein Zerfallsmass mitfuehren (leere Antworten, Laenge,
Wiederholungsschleifen). Das klammert ein, ab welcher Dosis blosse Beschaedigung
ueberhaupt etwas bewegt.

## Drei Ausgaenge, alle informativ

| Verdikt | Lesart |
|---|---|
| `INSTRUMENT-TRAEGT` | eine bessere Auswahl lohnt sich, v3 kann kommen |
| `AUSWAHL-OHNE-SCHAERFE` | Beschaedigung wirkt, gezielte Auswahl nicht - einzelne Einheiten sind das falsche Raster |
| `ABLATION-WIRKUNGSLOS` | das Verhalten ist gegen Ausfaelle dieser Groessenordnung unempfindlich |
| `NUR-STOERUNG` | auch der Zufallsarm senkt - Dosis zu hoch, keine Zuordnung moeglich |
| `EICHMARKE-FEHLT` | der JP-Arm folgt der Anweisung gar nicht - dann sagt die Kontrolle nichts |

**Vorregistriert:** gewaehlt werden Einheiten, die im JP-Zustand *hoeher* sind als im
NEU-Zustand. Ihre Ablation muss die Japanisch-Rate *senken*, die Zufallsauswahl derselben
Groesse darf das nicht. Alles andere ist ein Nein.

Die Zelle prueft bei jedem Eingriff direkt nach: sind die Einheiten wirklich null,
**wie viele** davon liessen sich nachmessen (nach dem Eingriff kann der Router andere
Experten waehlen), hat sich am Logit ueberhaupt etwas bewegt, und trifft die
Wiederherstellung den Ausgangszustand exakt.


In [ ]:
# === PHASE 12 - EICHUNG DES ABLATIONS-INSTRUMENTS ===========================
# Der FFN-Lauf v2 endete mit KEIN-EFFEKT: 64 ausgewaehlte Einheiten auf null zu
# setzen bewegte die Kipprate nicht (70.3% -> 81.2%, p=0.22), gleich viele
# zufaellige auch nicht (75.0%, p=0.69). Die Nachpruefungen waren sauber, der
# Eingriff hat also gegriffen. Trotzdem sagt dieser Befund NICHTS ueber das
# Modell, solange eine Frage offen ist:
#
#   KANN DAS AUSSCHALTEN EINZELNER FFN-EINHEITEN IN DIESEM MODELL UEBERHAUPT
#   EIN VERHALTEN BEWEGEN?
#
# Ist die Antwort nein, dann ist jeder Nullbefund - auch einer mit besserer
# Auswahl - ein Befund ueber das Werkzeug und nicht ueber die Sache. Deshalb
# wird hier nicht die Auswahl verfeinert, sondern das INSTRUMENT GEEICHT.
#
# ZWEI DEFEKTE AUS v2, DIE HIER BEHOBEN SIND
#
# (1) 'Trennwerte 31.50 .. 11523438.00'. In trenn_statistik stand
#     sd = sqrt(pooled_var) + 1e-8. Eine Einheit mit Varianz null in BEIDEN
#     Haelften - konstant in den hohen Praefixen, exakt 0 in den niedrigen,
#     weil der Router ihren Experten dort nicht waehlt - bekommt den Nenner
#     1e-8 und damit einen Trennwert von 1e7. Gewaehlt wurden also Einheiten
#     mit verschwindender Streuung, nicht mit grosser Trennung. Hier wird auf
#     EINEN GLOBALEN, robusten Massstab normiert (Median der Betraege ueber
#     alle Einheiten). Der kann nicht kollabieren, also kann kein Nenner eine
#     Trennung herbeizaubern.
#
# (2) 'aktive (Schicht,Experte)-Paare: 1010 | in JEDEM Praefix: 13'. Von 1010
#     Paaren waren 13 ueberall aktiv. Der Rest der Merkmalsmatrix war
#     STRUKTURELLE NULL aus unterschiedlichem Routing. Die Auswahl mass
#     'dieser Experte feuert hier und dort nicht' - bei fuenf verschiedenen
#     Zeichenketten trivial wahr und ohne Bezug zum Verhalten. Hier werden nur
#     Paare benutzt, die in ALLEN verglichenen Zustaenden aktiv sind.
#
# DER AUFBAU
#
# A POSITIVKONTROLLE. Zwei Arme desselben Prompts mit einem Verhaltensabstand,
#   der nicht strittig ist:
#       JP   "each service's Japanese name"   -> Antwort in japanischer Schrift
#       NEU  "each service's name"            -> Antwort englisch
#   Einheiten aus der Zustandsdifferenz waehlen (JP hoch gegen NEU), auf null
#   setzen, JAPANISCH-Rate neu messen. Kontrolle: gleich viele zufaellige
#   Einheiten aus derselben gemeinsamen Menge. Zwei Dosen, 64 und 512.
#   Das ist die Eichmarke: eine Anweisung, der das Modell zu ~100% folgt.
#
# B DOSISLEITER. Am urspruenglichen Prompt 64/256/1024 ZUFAELLIGE aktive
#   Einheiten ausschalten und neben der Kipprate ein Zerfallsmass mitfuehren
#   (leere Antworten, Laenge, Wiederholungsschleifen). Das klammert ein, ab
#   welcher Dosis blosse Beschaedigung ueberhaupt etwas bewegt.
#
# DREI AUSGAENGE, ALLE INFORMATIV
#   Instrument traegt          -> eine bessere Auswahl lohnt sich, v3 kann kommen
#   blind, aber Leiter wirkt   -> Einheiten-Ablation ist das falsche Raster
#   blind und Leiter still     -> das Verhalten ist massiv redundant und mit
#                                 Ablation in dieser Groessenordnung nicht pruefbar
#
# VORREGISTRIERT: gewaehlt werden Einheiten, die im JP-Zustand HOEHER sind als
# im NEU-Zustand. Ihre Ablation muss die JAPANISCH-Rate SENKEN. Die
# Zufallsauswahl derselben Groesse darf das nicht. Alles andere ist ein Nein.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_ffn_eichung")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
JPW=[(0x3040,0x30FF),(0x3400,0x9FFF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _in(c,bereiche):
    o=ord(c); return any(a<=o<=b for a,b in bereiche)
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250 and _in(c,FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and _in(ch,FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def ist_jp(t,mindest=3):
    """Eigener Zaehler fuer die Positivkontrolle. Im JP-Arm ist japanische
       Schrift das ERWUENSCHTE Verhalten - classify_breit wuerde sie
       'takeover' nennen, was hier irrefuehrend waere. Gezaehlt wird ein Lauf
       von mindestens 3 Kana-/Kanji-Zeichen: einzelne Zeichen kommen auch in
       englischen Antworten als Beispiel vor, ein Lauf nicht."""
    c=0
    for ch in t:
        if _in(ch,JPW):
            c+=1
            if c>=mindest: return True
        elif ch.isalpha(): c=0
    return False
def wiederholt(t,fenster=12,mal=4):
    """Zerfallsmerkmal: dieselbe Zeichenfolge viermal. Bei starker Beschaedigung
       faellt ein Modell in Schleifen, lange bevor es verstummt."""
    if len(t)<fenster*mal: return False
    z=collections.Counter(t[i:i+fenster] for i in range(len(t)-fenster+1))
    return max(z.values())>=mal
def zerfall(texte):
    """Was sagt die Antwortform ueber den Schaden - unabhaengig von der Sprache"""
    if not texte: return dict(leer=0.0,laenge=0.0,schleife=0.0)
    return dict(leer=sum(1 for t in texte if not t.strip())/len(texte),
                laenge=sum(len(t) for t in texte)/len(texte),
                schleife=sum(1 for t in texte if wiederholt(t))/len(texte))
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def phrase_mit(w):
    return "each service's name" if not w else "each service's %s name"%w
def setze_arm(text,ersatz):
    if PHRASE not in text: return text,False
    return text.replace(PHRASE,ersatz),True
def massstab(*vektoren):
    """EIN globaler, robuster Massstab fuer alle Einheiten. Ersetzt den
       Nenner je Einheit aus v2, der auf 1e-8 fallen konnte und damit
       Trennwerte von 1e7 erzeugt hat. Median statt Mittelwert, weil die
       Zwischenschicht duennbesetzt ist und wenige grosse Werte den Mittelwert
       tragen wuerden. Nullen zaehlen NICHT mit: bei 70% strukturellen Nullen
       waere der Median sonst selbst null."""
    v=np.abs(np.concatenate([np.asarray(x,dtype=np.float64).ravel() for x in vektoren]))
    v=v[v>0]
    if v.size==0: return 1.0
    m=float(np.median(v))
    return m if m>0 else 1.0
def trennung_zwei(xa,xb,s):
    """Differenz zweier Zustaende in Einheiten EINES globalen Massstabs.
       Positiv = im ersten Zustand hoeher. Beschraenkt und vergleichbar."""
    return (np.asarray(xa,dtype=np.float64)-np.asarray(xb,dtype=np.float64))/float(s)
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: im JP-Zustand hoeher, Ablation muss die JP-Rate senken"""
    return list(np.argsort(-np.asarray(d))[:k])
def zu_einheit(i,paare,breite):
    """flacher Index -> ((Schicht,Experte), Einheit). Die flache Anordnung ist
       genau die aus np.concatenate ueber die Paare in DIESER Reihenfolge -
       geht die Zuordnung schief, ablatiert man die falschen Zeilen und merkt
       es nirgends."""
    return paare[i//breite],i%breite
def gemeinsame_paare(zustaende):
    """nur (Schicht,Experte), die in ALLEN Zustaenden aktiv sind - alles andere
       ist strukturelle Null aus unterschiedlichem Routing und traegt keine
       Information ueber Staerke"""
    if not zustaende: return []
    g=set(zustaende[0])
    for z in zustaende[1:]: g&=set(z)
    return sorted(g)
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    """senkt / hebt / still - zweiseitig, weil die Leiter keine Richtung
       vorregistriert: sie fragt nur, ob sich UEBERHAUPT etwas bewegt"""
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_arm(k_bas,n_bas,k_aus,n_aus,k_zuf,n_zuf,alpha=0.05):
    a=urteil_dosis(k_bas,n_bas,k_aus,n_aus,alpha)=="senkt"
    z=urteil_dosis(k_bas,n_bas,k_zuf,n_zuf,alpha)=="senkt"
    if a and not z: return "TRAEGT"
    if a and z:     return "NUR-STOERUNG"
    if z and not a: return "WIDERSPRUECHLICH"
    return "BLIND"
def urteil_eichung(arm_urteile,leiter_urteile,jp_basisrate,mindestrate=0.5):
    """Gesamturteil. Zuerst der Sanitaetscheck: wenn der JP-Arm ohne Eingriff
       gar nicht japanisch antwortet, gibt es nichts zu senken und die
       Positivkontrolle ist ungueltig - das muss man sagen, statt es als
       'blind' zu verbuchen."""
    if jp_basisrate<mindestrate: return "EICHMARKE-FEHLT"
    if "TRAEGT" in arm_urteile: return "INSTRUMENT-TRAEGT"
    if "NUR-STOERUNG" in arm_urteile: return "NUR-STOERUNG"
    if any(u!="still" for u in leiter_urteile): return "AUSWAHL-OHNE-SCHAERFE"
    return "ABLATION-WIRKUNGSLOS"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",48)); N_LEIT=int(globals().get("N_LEIT",48))
MAX_NEW=int(globals().get("MAX_NEW",64)); CHUNK=int(globals().get("CHUNK",16))
TEMP=float(globals().get("TEMP",1.0)); SEED=int(globals().get("SEED",20260809))
DOSEN=list(globals().get("DOSEN",[64,512])); LEITER=list(globals().get("LEITER",[64,256,1024]))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
ROH_PROMPT=PROMPTS[ZIEL_ID]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
EXPM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; DP=e0.down_proj; INTER=int(e0.intermediate_dim)
    ARCH_OK=(GU.ndim==3 and DP.ndim==3 and GU.shape[1]==2*INTER
             and GU.shape[2]==cfg.hidden_size and DP.shape[2]==INTER)
    print("  %d Schichten | gate_up_proj %s | down_proj %s | Zwischenbreite %d"
          %(len(EXPM),tuple(GU.shape),tuple(DP.shape),INTER))
    print("  aktiv je Token %d x %d = %d Einheiten | Formen wie erwartet: %s"
          %(cfg.num_experts_per_tok,INTER,cfg.num_experts_per_tok*INTER,
            "ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    EICH_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- Werkzeuge ---------------------------------------------------
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
def zwischenschicht(mod,x,e):
    """exakt die zwei Zeilen aus Qwen3_5MoeExperts.forward"""
    gu=torch.nn.functional.linear(x,mod.gate_up_proj[e])
    g,u=gu.chunk(2,dim=-1)
    return (mod.act_fn(g)*u)
def hole_zustand(text):
    """Zustand am LETZTEN Token. Der ist in allen Armen derselbe Token - nur
       der Zusammenhang davor unterscheidet sich. Genau das soll verglichen
       werden."""
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l):
        def h(mod,args):
            fang[l]=(args[0].detach(),args[1].detach())
            return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear()
            o2=model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
            lg=o2.logits[0,-1].float().cpu().numpy()
    finally:
        for h in hs: h.remove()
    zust={}
    with torch.no_grad():
        for l,(x,idx) in fang.items():
            xv=x.reshape(-1,x.shape[-1])[-1]
            ii=idx.reshape(-1,idx.shape[-1])[-1]
            for e in ii.tolist():
                zust[(l,int(e))]=zwischenschicht(EXPM[l],xv,int(e)).float().cpu().numpy()
    return zust,lg
def setze_null(einheiten):
    """gate- UND up-Zeile auf null -> act_fn(0)*0 = 0. Nur die geaenderten
       Zeilen werden gesichert, nicht die 1 GB Matrix."""
    sicher=[]
    with torch.no_grad():
        for (l,e),u in einheiten:
            W=EXPM[l].gate_up_proj
            for r in (u,INTER+u):
                sicher.append((l,e,r,W[e,r].clone())); W[e,r].zero_()
    return sicher
def stelle_her(sicher):
    with torch.no_grad():
        for l,e,r,v in sicher: EXPM[l].gate_up_proj[e,r].copy_(v)
def mit_ablation(einheiten,text,n,startwert,pruef_text=None,pruef_logit=None):
    """ablatieren, ziehen, IMMER zuruecksetzen. Liefert Texte und die
       Nachpruefzahlen: sind die Einheiten wirklich null, wie viele davon
       liessen sich nachmessen, und hat sich am Logit ueberhaupt etwas bewegt.
       Die Zahl der nachgemessenen ist wichtig: nach dem Eingriff kann der
       Router an derselben Stelle ANDERE Experten waehlen - dann ist eine
       Einheit nicht mehr im aufgezeichneten Zustand und 'rest=0' waere sonst
       nur die leere Menge."""
    sicher=setze_null(einheiten); rest=0.0; wirk=float("nan"); nach=0
    try:
        if pruef_text is not None:
            z2,lg2=hole_zustand(pruef_text)
            tot=[abs(float(z2[q][u])) for q,u in einheiten if q in z2]
            nach=len(tot); rest=max(tot) if tot else 0.0
            if pruef_logit is not None: wirk=float(np.abs(lg2-pruef_logit).max())
        aus=zieh(text,n,startwert)
    finally:
        stelle_her(sicher)
    return aus,rest,wirk,nach
def zeile(nm,k,n,kb,nb,extra=""):
    pp,lo,hi=wilson(k,n)
    pv="-" if kb is None else "%.4f"%fisher2x2(k,n-k,kb,nb-kb)
    print("  %-28s %3d/%-4d %5.1f%% [%4.1f,%4.1f] %9s  %s"
          %(nm,k,n,100*pp,100*lo,100*hi,pv,extra))
def zf(texte):
    z=zerfall(texte)
    return "leer %.0f%% | %3.0f Zeichen | Schleife %.0f%%"%(100*z["leer"],z["laenge"],
                                                            100*z["schleife"])
# ---------------- A  Positivkontrolle: Grundraten -----------------------------
print(""); print("="*80); print("A  POSITIVKONTROLLE - EICHMARKE MESSEN"); print("="*80)
JP_ROH,ok_j=setze_arm(ROH_PROMPT,phrase_mit("Japanese"))
NE_ROH,ok_n=setze_arm(ROH_PROMPT,phrase_mit(""))
assert ok_j and ok_n,"Phrase %r nicht im Prompt gefunden"%PHRASE
JP=prompt_text(JP_ROH); NE=prompt_text(NE_ROH); OR=prompt_text(ROH_PROMPT)
print("  JP  ...%s..."%phrase_mit("Japanese"))
print("  NEU ...%s..."%phrase_mit(""))
t0=time.time()
A_JP=zieh(JP,N_ARM,SEED+1); A_NE=zieh(NE,N_ARM,SEED+2); A_OR=zieh(OR,N_ARM,SEED+3)
K_JP=sum(ist_jp(t) for t in A_JP); K_NE=sum(ist_jp(t) for t in A_NE)
K_OR=sum(classify_breit(t) in SWB for t in A_OR)
print("  (%.0f s)"%(time.time()-t0)); print("")
print("  %-28s %8s %7s %20s %9s"%("Arm","k/n","Rate","95%-Intervall","p vs Basis"))
zeile("JP  japanische Schrift",K_JP,N_ARM,None,None,zf(A_JP))
zeile("NEU japanische Schrift",K_NE,N_ARM,None,None,zf(A_NE))
zeile("ORIG Kippen (breit)",K_OR,N_ARM,None,None,zf(A_OR))
# ---------------- A  Zustaende und Auswahl ------------------------------------
print(""); print("-"*80); print("A  ZUSTAENDE VERGLEICHEN")
Z_JP,L_JP=hole_zustand(JP); Z_NE,L_NE=hole_zustand(NE); Z_OR,L_OR=hole_zustand(OR)
GEM=gemeinsame_paare([Z_JP,Z_NE])
print("  aktive Paare: JP %d | NEU %d | in BEIDEN %d"%(len(Z_JP),len(Z_NE),len(GEM)))
assert len(GEM)>=2,"zu wenige gemeinsame Experten fuer einen Vergleich"
XJ=np.concatenate([Z_JP[q] for q in GEM]).astype(np.float64)
XN=np.concatenate([Z_NE[q] for q in GEM]).astype(np.float64)
S=massstab(XJ,XN)
D=trennung_zwei(XJ,XN,S)
print("  gemeinsame Einheiten %d | Massstab (Median |x|, ohne Nullen) %.4f"%(len(D),S))
print("  Trennung: Median %.3f | 99%%-Quantil %.3f | Maximum %.3f"
      %(float(np.median(D)),float(np.percentile(D,99)),float(D.max())))
print("  (in v2 stand hier 1.15e7 - der Nenner war je Einheit und konnte auf 1e-8 fallen)")
AKTIV_GEM=[i for i in range(len(D)) if max(abs(XJ[i]),abs(XN[i]))>1e-3]
print("  davon aktiv (|x|>1e-3 in mindestens einem Arm): %d"%len(AKTIV_GEM))
rnd=random.Random(SEED)
# ---------------- A  Der Test je Dosis ----------------------------------------
print(""); print("-"*80); print("A  ABLATION IM JP-ARM (vorregistriert: muss SENKEN)")
ARM_URTEILE=[]; ARM_ZEILEN=[]
for K in DOSEN:
    if K>len(AKTIV_GEM):
        print("  Dosis %d uebersprungen - nur %d gemeinsame aktive Einheiten"%(K,len(AKTIV_GEM)))
        continue
    idx=waehle_einheiten(D,K)
    AUSW=[zu_einheit(i,GEM,INTER) for i in idx]
    ZUFI=rnd.sample(AKTIV_GEM,K)
    ZUFA=[zu_einheit(i,GEM,INTER) for i in ZUFI]
    a,rest_a,wirk_a,na=mit_ablation(AUSW,JP,N_ARM,SEED+100+K,JP,L_JP)
    z,rest_z,wirk_z,nz=mit_ablation(ZUFA,JP,N_ARM,SEED+200+K,JP,L_JP)
    ka=sum(ist_jp(t) for t in a); kz=sum(ist_jp(t) for t in z)
    u=urteil_arm(K_JP,N_ARM,ka,N_ARM,kz,N_ARM); ARM_URTEILE.append(u)
    print("")
    print("  DOSIS %d  Trennwerte %.3f .. %.3f"%(K,D[idx[-1]],D[idx[0]]))
    print("  NACHPRUEFUNG  nachgemessen %d/%d bzw. %d/%d | groesster Rest %.2e / %.2e "
          "| Logit bewegt %.4f / %.4f"%(na,K,nz,K,rest_a,rest_z,wirk_a,wirk_z))
    assert max(rest_a,rest_z)<1e-6,"Einheit nach der Ablation nicht null"
    assert max(wirk_a,wirk_z)>1e-3,"Ablation ohne jede Logit-Wirkung"
    assert min(na,nz)>0,"keine einzige Einheit nachmessbar - Nachpruefung waere leer"
    zeile("ohne Eingriff",K_JP,N_ARM,None,None,zf(A_JP))
    zeile("gewaehlte %d aus"%K,ka,N_ARM,K_JP,N_ARM,zf(a))
    zeile("zufaellige %d aus"%K,kz,N_ARM,K_JP,N_ARM,zf(z))
    print("  -> %s"%u)
    ARM_ZEILEN.append(dict(dosis=K,k_ausw=ka,k_zufall=kz,n=N_ARM,urteil=u,
                           trenn_min=float(D[idx[-1]]),trenn_max=float(D[idx[0]]),
                           zerfall_ausw=zerfall(a),zerfall_zufall=zerfall(z)))
# ---------------- B  Dosisleiter ----------------------------------------------
print(""); print("="*80); print("B  DOSISLEITER - ZUFAELLIGE EINHEITEN AM URSPRUENGLICHEN PROMPT")
print("="*80)
PAARE_OR=sorted(Z_OR)
AKTIV_OR=[(q,u) for q in PAARE_OR for u in range(INTER) if abs(Z_OR[q][u])>1e-3]
print("  aktive Einheiten im ORIG-Zustand: %d (von %d x %d aufgezeichneten)"
      %(len(AKTIV_OR),len(PAARE_OR),INTER))
print("  Achtung: ausgeschaltet wird ueberall, gemessen wird an EINER Stelle -")
print("  bei der Erzeugung routet das Modell an anderen Stellen zu anderen Experten.")
print("")
zeile("ohne Eingriff",K_OR,N_ARM,None,None,zf(A_OR))
LEITER_URTEILE=[]; LEITER_ZEILEN=[]
rnd2=random.Random(SEED+999)
for K in LEITER:
    if K>len(AKTIV_OR):
        print("  Stufe %d uebersprungen - nur %d aktive Einheiten"%(K,len(AKTIV_OR)))
        continue
    E=rnd2.sample(AKTIV_OR,K)
    a,rest,wirk,nk=mit_ablation(E,OR,N_LEIT,SEED+300+K,OR,L_OR)
    assert rest<1e-6,"Einheit nach der Ablation nicht null"
    k=sum(classify_breit(t) in SWB for t in a)
    u=urteil_dosis(K_OR,N_ARM,k,N_LEIT); LEITER_URTEILE.append(u)
    zeile("%d zufaellige aus"%K,k,N_LEIT,K_OR,N_ARM,zf(a))
    print("  %-28s nachgemessen %d/%d, Logit bewegt %.4f -> %s"%("",nk,K,wirk,u))
    LEITER_ZEILEN.append(dict(dosis=K,k=k,n=N_LEIT,urteil=u,logit=wirk,
                              nachgemessen=nk,zerfall=zerfall(a)))
# ---------------- Wiederherstellung pruefen -----------------------------------
z3,lg3=hole_zustand(JP)
print(""); print("  WIEDERHERSTELLUNG: groesste Logit-Abweichung zum Ausgangszustand %.2e"
                 %float(np.abs(lg3-L_JP).max()))
# ---------------- Urteil ------------------------------------------------------
CODE=urteil_eichung(ARM_URTEILE,LEITER_URTEILE,K_JP/max(N_ARM,1))
print(""); print("="*80); print("VERDIKT: %s"%CODE); print("="*80)
if CODE=="EICHMARKE-FEHLT":
    print("  Der JP-Arm antwortet ohne Eingriff nicht japanisch genug (%.0f%%). Dann"%(100*K_JP/N_ARM))
    print("  gibt es nichts zu senken und die Positivkontrolle sagt nichts. Zuerst")
    print("  einen Arm finden, dem das Modell zuverlaessig folgt.")
elif CODE=="INSTRUMENT-TRAEGT":
    print("  Das Ausschalten gezielt gewaehlter FFN-Einheiten senkt eine Anweisung,")
    print("  der das Modell sonst folgt - die Zufallsauswahl derselben Groesse nicht.")
    print("  Das Werkzeug hat Aufloesungsvermoegen. Damit wird der Nullbefund aus v2")
    print("  zu einer AUSSAGE UEBER DIE SACHE: dort war die Auswahl schlecht, nicht")
    print("  das Verfahren. Naechster Schritt: dieselbe Auswahl mit vielen Praefixen.")
elif CODE=="NUR-STOERUNG":
    print("  Auch zufaellige Einheiten senken die Eichmarke. Dann wirkt Beschaedigung")
    print("  und nicht Auswahl - auf dieser Dosis ist keine Zuordnung moeglich.")
    print("  Naechster Schritt: kleinere Dosis, bis der Zufallsarm still ist.")
elif CODE=="AUSWAHL-OHNE-SCHAERFE":
    print("  Zufaellige Beschaedigung bewegt das Verhalten, gezielte Auswahl nicht.")
    print("  Einzelne Einheiten sind damit das falsche Raster: was das Modell")
    print("  unterscheidet, liegt nicht in einer benennbaren Handvoll von ihnen.")
    print("  Das ist ein Befund ueber die Darstellung, kein Fehlschlag.")
else:
    print("  Weder gezielte noch zufaellige Ablation bewegt etwas - bis %d Einheiten."%max(LEITER))
    print("  Das Verhalten ist gegen Ausfaelle dieser Groessenordnung unempfindlich.")
    print("  Ablation einzelner Einheiten kann die Frage nicht beantworten, und der")
    print("  Nullbefund aus v2 ist damit ein Befund ueber das Werkzeug.")
EICH_RESULTS=dict(verdict=CODE,arch_ok=True,prompt_id=ZIEL_ID,inter=INTER,
    n_arm=N_ARM,n_leiter=N_LEIT,dosen=DOSEN,leiter=LEITER,
    k_jp=K_JP,k_neu=K_NE,k_orig=K_OR,
    zerfall_jp=zerfall(A_JP),zerfall_neu=zerfall(A_NE),zerfall_orig=zerfall(A_OR),
    paare_jp=len(Z_JP),paare_neu=len(Z_NE),paare_gemeinsam=len(GEM),
    massstab=S,einheiten_gemeinsam=int(len(D)),einheiten_aktiv=len(AKTIV_GEM),
    trenn_median=float(np.median(D)),trenn_max=float(D.max()),
    arme=ARM_ZEILEN,leiter_zeilen=LEITER_ZEILEN)
wc_save("antworten_eichung",dict(prompt_id=ZIEL_ID,jp=A_JP,neu=A_NE,orig=A_OR))
wc_save_all()
print("")
print("(Die Positivkontrolle ist der eigentliche Inhalt dieses Laufs. Sie sagt nicht,")
print(" wo das Sprachkippen sitzt - sie sagt, ob die Frage mit diesem Werkzeug")
print(" ueberhaupt beantwortbar ist. Ein Nein ist hier genauso brauchbar wie ein Ja.)")
